# UAVIDS-2025 — tuning aninhado em S2

Esta rodada testa se um espaço pequeno e congelado de hiperparâmetros melhora Random Forest e XGBoost quando cada escolha é feita somente em validação interna por grupos de origem. O teste externo de cada fold não participa da escolha.

O estudo permanece exploratório porque as famílias foram priorizadas após observar `predictive_baseline_v1`.


## Método

O [protocolo v2](../../protocol/nested_tuning_v2.md) e a [configuração](../../configs/nested_tuning_v2.json) definem três candidatos por família, três folds internos `StratifiedGroupKFold`, F1-macro como critério e desempate por complexidade. Em cada fold externo, nove modelos são ajustados internamente e o candidato escolhido é reajustado no treino externo completo.


In [1]:
from pathlib import Path
import json
import pandas as pd

project_root = Path.cwd().resolve()
if project_root.name == "research":
    project_root = project_root.parents[1]
elif project_root.name == "notebooks":
    project_root = project_root.parent

result_dir = project_root / "results" / "nested_tuning_v2"
report_dir = project_root / "reports" / "nested_tuning_v2"
manifest = json.loads((result_dir / "experiment_manifest.json").read_text("utf-8"))
print(json.dumps(manifest, indent=2, ensure_ascii=False))


{
  "experiment_id": "nested_tuning_v2",
  "status": "exploratory_nested_tuning_after_baseline_v1",
  "config_sha256": "46a0068f25241ce2297d189b4793ff6a2bd1c2de4eba5274a7f5aae74d0b3070",
  "dataset_sha256": "d50d339f68be7b23f0bf089dd438b20a1835c13182d8641220538121440164d0",
  "split_sha256": "352d1966e2dda8060a7a59d69eafa49c461e4c2415db292319e0e1526d2996cc",
  "completed_jobs": 10,
  "expected_jobs": 10,
  "complete": true,
  "environment": {
    "python": "3.12.10",
    "platform": "Windows-11-10.0.26200-SP0",
    "numpy": "1.26.4",
    "pandas": "2.2.3",
    "scikit_learn": "1.5.2",
    "xgboost": "2.0.3",
    "processor": "AMD64 Family 25 Model 33 Stepping 2, AuthenticAMD",
    "logical_cpu_count": 12,
    "threads_allowed_per_fit": 4,
    "accelerator": "none_used"
  },
  "class_order": [
    "Normal Traffic",
    "Blackhole Attack",
    "Flooding Attack",
    "Sybil Attack",
    "Wormhole Attack"
  ],
  "prediction_schema": [
    "row_id",
    "flow_id",
    "protocol",
    "fold"

## Seleção interna e custo

In [2]:
selections = pd.read_csv(report_dir / "selections_by_fold.csv")
print(selections.round(6).to_string(index=False))


 fold         model selected_candidate_id  inner_selection_f1_macro  outer_f1_macro  tuning_fit_count  tuning_seconds  final_fit_seconds  warning_count
    0 random_forest     rf_240_sqrt_leaf2                  0.947195        0.947586                 9      135.266796          14.977775              0
    1 random_forest     rf_240_sqrt_leaf2                  0.948862        0.953166                 9      103.988193          14.569636              0
    2 random_forest     rf_240_sqrt_leaf2                  0.938771        0.955152                 9       99.117540          14.309071              0
    3 random_forest     rf_160_sqrt_leaf1                  0.949964        0.941964                 9      116.131702          10.814087              0
    4 random_forest     rf_240_sqrt_leaf2                  0.946176        0.954768                 9       98.340397          13.541527              0
    0       xgboost   xgb_200_depth6_lr10                  0.946964        0.948015     

In [3]:
candidate_scores = pd.read_csv(report_dir / "inner_candidate_scores.csv")
print(candidate_scores.round(6).to_string(index=False))


 fold         model                    candidate_id  complexity_rank  mean_inner_f1_macro  std_inner_f1_macro  selected
    0 random_forest               rf_160_sqrt_leaf1                1             0.946830            0.003513     False
    0 random_forest               rf_240_half_leaf1                3             0.946799            0.003653     False
    0 random_forest               rf_240_sqrt_leaf2                2             0.947195            0.002266      True
    1 random_forest               rf_160_sqrt_leaf1                1             0.947466            0.003651     False
    1 random_forest               rf_240_half_leaf1                3             0.946462            0.003719     False
    1 random_forest               rf_240_sqrt_leaf2                2             0.948862            0.003330      True
    2 random_forest               rf_160_sqrt_leaf1                1             0.937966            0.010378     False
    2 random_forest               rf_240

## Comparação pareada com o baseline

Cada diferença abaixo usa o mesmo fold externo S2. Ela descreve o efeito observado do procedimento de tuning; os cinco folds não são tratados como cinco datasets independentes.


In [4]:
comparison = pd.read_csv(report_dir / "baseline_vs_tuning.csv")
columns = ["fold", "model", "baseline_f1_macro", "tuned_f1_macro", "f1_difference"]
print(comparison[columns].round(6).to_string(index=False))


 fold         model  baseline_f1_macro  tuned_f1_macro  f1_difference
    0 random_forest           0.946797        0.947586       0.000789
    1 random_forest           0.950961        0.953166       0.002205
    2 random_forest           0.954480        0.955152       0.000672
    3 random_forest           0.941944        0.941964       0.000020
    4 random_forest           0.953819        0.954768       0.000949
    0       xgboost           0.948015        0.948015       0.000000
    1       xgboost           0.951935        0.952241       0.000306
    2       xgboost           0.955278        0.956111       0.000832
    3       xgboost           0.945997        0.945245      -0.000752
    4       xgboost           0.956898        0.956047      -0.000851


![Efeito pareado do tuning](../../reports/nested_tuning_v2/paired_tuning_effect.png)

In [5]:
effect_summary = pd.read_csv(report_dir / "tuning_effect_summary.csv")
print(effect_summary.round(6).to_string(index=False))


        model  baseline_f1_macro__mean  baseline_f1_macro__std  baseline_f1_macro__min  baseline_f1_macro__max  tuned_f1_macro__mean  tuned_f1_macro__std  tuned_f1_macro__min  tuned_f1_macro__max  f1_difference__mean  f1_difference__std  f1_difference__min  f1_difference__max
random_forest                 0.949600                0.005243                0.941944                0.954480              0.950527             0.005664             0.941964             0.955152             0.000927            0.000797            0.000020            0.002205
      xgboost                 0.951625                0.004635                0.945997                0.956898              0.951532             0.004841             0.945245             0.956111            -0.000093            0.000713           -0.000851            0.000832


## Interpretação

- O RF melhorou em média **+0.000927** de F1-macro por fold.
- O XGBoost mudou **-0.000093**, sem ganho prático.
- O candidato XGBoost base venceu internamente nos cinco folds. Isso indica que aumentar a complexidade dentro do espaço registrado não foi necessário.
- O RF mais regularizado venceu quatro folds, mas a melhora média ficou abaixo de 0,001.
- As diferenças entre folds são maiores que o ganho médio do tuning. A composição dos grupos de origem continua sendo a principal fonte visível de variação.

Não ampliaremos a busca com base nesses testes. Para o benchmark de sistemas, a configuração XGBoost base pode ser congelada como candidata principal e o RF ajustado como comparador. Antes de conclusões preditivas finais, ainda são necessárias múltiplas sementes e, idealmente, novas simulações externas.
